# Federal Reserve Sentiment Analysis - Example Notebook

This notebook demonstrates how to use the Fed Sentiment Analysis system to:
1. Scrape Twitter for economy-related tweets
2. Analyze sentiment using LLMs
3. Track Federal Reserve decisions
4. Predict upcoming decisions based on sentiment
5. Visualize results

In [ ]:
import sys
sys.path.append('..')

from src.utils import Database, load_config
from src.utils.config_loader import load_env
from src.scraper import TwitterScraper
from src.sentiment import SentimentAnalyzer
from src.fed_tracker import FedTracker
from src.analysis import CorrelationAnalyzer, FedPredictor
from src.visualization import Visualizer

import pandas as pd
from datetime import datetime, timedelta

# Load environment variables
load_env()

# Initialize database
db = Database()

## 1. Scrape Twitter for Economy-Related Tweets

In [ ]:
# Initialize scraper
scraper = TwitterScraper()

# Run scrape
tweets = scraper.run_full_scrape()

print(f"Collected {len(tweets)} tweets")

# Save to database
for tweet in tweets:
    db.insert_tweet(tweet)

print("Tweets saved to database")

## 2. Analyze Sentiment with LLM

In [ ]:
# Get unanalyzed tweets
unanalyzed = db.get_tweets_for_sentiment_analysis(limit=20)

print(f"Analyzing {len(unanalyzed)} tweets...")

# Initialize analyzer
analyzer = SentimentAnalyzer()

# Analyze tweets
sentiments = analyzer.analyze_tweets_batch(unanalyzed)

# Save results
for sentiment in sentiments:
    db.insert_sentiment(sentiment)

# Get aggregate statistics
aggregate = analyzer.get_aggregate_sentiment(sentiments)
print("\nSentiment Summary:")
print(f"Average Sentiment: {aggregate['avg_sentiment']:.3f}")
print(f"Bullish: {aggregate['bullish_percentage']:.1f}%")
print(f"Bearish: {aggregate['bearish_percentage']:.1f}%")
print(f"Neutral: {aggregate['neutral_percentage']:.1f}%")

## 3. Track Federal Reserve Information

In [ ]:
# Initialize Fed tracker
fed_tracker = FedTracker()

# Get current Fed rate
current_rate = fed_tracker.get_current_fed_rate()
print(f"Current Fed Funds Rate: {current_rate}%")

# Get next meeting
next_meeting = fed_tracker.get_next_meeting()
if next_meeting:
    print(f"Next FOMC Meeting: {next_meeting['date']} ({next_meeting['days_until']} days away)")

# Get economic indicators
indicators = fed_tracker.get_economic_indicators()
print("\nEconomic Indicators:")
for name, data in indicators.items():
    print(f"  {name}: {data['value']} (as of {data['date']})")

## 4. Make Prediction for Next Meeting

In [ ]:
# Initialize predictor
predictor = FedPredictor(db)

# Get next meeting date
next_meeting = fed_tracker.get_next_meeting()

if next_meeting:
    meeting_date = next_meeting['date']

    # Generate prediction report
    prediction = predictor.generate_prediction_report(meeting_date)

    print("\nPREDICTION REPORT")
    print("="*50)
    print(f"Meeting Date: {meeting_date}")
    print(f"Predicted Action: {prediction['predicted_action'].replace('_', ' ').title()}")
    print(f"Confidence: {prediction['confidence']*100:.1f}%")
    print(f"\nSentiment Metrics:")
    print(f"  Average: {prediction['sentiment_avg']:.3f}")
    print(f"  Recent: {prediction['sentiment_recent']:.3f}")
    print(f"  Trend: {prediction['sentiment_trend']:.3f}")
    print(f"  Bullish: {prediction['bullish_percentage']:.1f}%")
    print(f"  Bearish: {prediction['bearish_percentage']:.1f}%")
    print(f"\nReasoning: {prediction['reasoning']}")
    print("="*50)

## 5. Visualize Results

In [ ]:
# Initialize visualizer
visualizer = Visualizer()

# Get sentiment data
end_date = datetime.now().strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d")

correlation_analyzer = CorrelationAnalyzer(db)
sentiment_df = correlation_analyzer.prepare_sentiment_data(start_date, end_date)

if not sentiment_df.empty:
    # Plot sentiment trend
    visualizer.plot_sentiment_trend(sentiment_df, title="Twitter Sentiment - Last 30 Days")

    # Plot sentiment distribution
    visualizer.plot_sentiment_distribution(sentiments)

    # Create prediction summary chart
    if next_meeting:
        visualizer.create_prediction_summary_chart(prediction)

print("Visualizations created in data/plots/")

## 6. Analyze Historical Correlation

In [ ]:
# Get historical Fed decisions
end_date = datetime.now().strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=365)).strftime("%Y-%m-%d")

decisions = fed_tracker.get_historical_decisions(start_date, end_date)

# Analyze correlation
correlation_result = correlation_analyzer.correlate_sentiment_with_decisions(decisions)

print("\nCORRELATION ANALYSIS")
print("="*50)
print(f"Sample Size: {correlation_result['sample_size']} meetings")

if correlation_result['correlation'] is not None:
    print(f"Correlation: {correlation_result['correlation']:.3f}")
    print(f"P-Value: {correlation_result['p_value']:.4f}")
    print(f"Significant: {correlation_result['statistically_significant']}")
    print(f"\n{correlation_result['interpretation']}")
else:
    print(correlation_result['message'])

print("="*50)